# Langkah 8: Sistem Rekomendasi Film Berbasis TransH
Skripsi: Sistem Rekomendasi Film Berbasis TransH Knowledge Graph Embedding

**Input** (dari `07_output/`):
- `model_transh.pt` — bobot model TransH
- `entity_embeddings.npy`, `relation_embeddings.npy`, `hyperplane_normals.npy`
- `entity2id.json`, `id2entity.json`, `relation2id.json`, `id2relation.json`
- `06_split/` — data train/valid/test untuk evaluasi
- `DATA_FILM_with_USER/liked_netflix_movies.csv` — metadata film (judul, genre, dll.)

**Output** (disimpan ke `08_output/`):
- `rekomendasi_per_user.csv` — Top-N rekomendasi untuk setiap user
- `evaluasi_rekomendasi.csv` — metrik Precision/Recall/NDCG per user
- `evaluasi_summary.txt` — ringkasan metrik agregat

### Cara Kerja Rekomendasi
Untuk user `U_<id>`, kita cari film `s<id>` dengan skor terkecil:
```
score(U_x, liked, s_y) = ||U_x_⊥ + d_liked − s_y_⊥||²
```
Film dengan skor terkecil = paling direkomendasikan.

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# ── Path ──
BASE_DIR   = Path().resolve()
MODEL_DIR  = BASE_DIR / "07_output"
SPLIT_DIR  = BASE_DIR / "06_split"
FILM_CSV   = BASE_DIR / "DATA_FILM_with_USER" / "liked_netflix_movies.csv"
OUT_DIR    = BASE_DIR / "08_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("  Sistem Rekomendasi Film — TransH Embedding")
print("=" * 60)
print(f"  Model dir : {MODEL_DIR}")
print(f"  Output dir: {OUT_DIR}")
print(f"  Device    : {DEVICE}")

---
## A. Load Model & Vocab

In [ ]:
# ── Load vocab ──
with open(MODEL_DIR / "entity2id.json",   "r", encoding="utf-8") as f:
    entity2id = json.load(f)
with open(MODEL_DIR / "id2entity.json",   "r", encoding="utf-8") as f:
    id2entity = {int(k): v for k, v in json.load(f).items()}
with open(MODEL_DIR / "relation2id.json", "r", encoding="utf-8") as f:
    relation2id = json.load(f)
with open(MODEL_DIR / "id2relation.json", "r", encoding="utf-8") as f:
    id2relation = {int(k): v for k, v in json.load(f).items()}

n_entities  = len(entity2id)
n_relations = len(relation2id)

print(f"Entitas  : {n_entities:,}")
print(f"Relasi   : {n_relations}")
print(f"  Daftar relasi: {list(relation2id.keys())}")

In [ ]:
# ── Definisi ulang class TransH (agar bisa load model) ──
class TransH(nn.Module):
    def __init__(self, n_ent, n_rel, dim=128, margin=1.5, C=0.001, eps=1e-5):
        super().__init__()
        self.dim    = dim
        self.margin = margin
        self.C      = C
        self.eps    = eps
        self.ent_emb  = nn.Embedding(n_ent, dim)
        self.rel_emb  = nn.Embedding(n_rel, dim)
        self.norm_emb = nn.Embedding(n_rel, dim)

    def _project(self, e, w):
        return e - (e * w).sum(dim=-1, keepdim=True) * w

    def score(self, h_ids, r_ids, t_ids):
        h = self.ent_emb(h_ids)
        t = self.ent_emb(t_ids)
        d = self.rel_emb(r_ids)
        w = nn.functional.normalize(self.norm_emb(r_ids), dim=-1)
        h_perp = nn.functional.normalize(self._project(h, w), dim=-1)
        t_perp = nn.functional.normalize(self._project(t, w), dim=-1)
        return torch.norm(h_perp + d - t_perp, p=2, dim=-1)

# ── Load checkpoint ──
checkpoint = torch.load(MODEL_DIR / "model_transh.pt", map_location=DEVICE)
hp = checkpoint["hyperparams"]

model = TransH(
    n_ent=hp["n_entities"],
    n_rel=hp["n_relations"],
    dim=hp["dim"],
    margin=hp["margin"],
    C=hp["C"],
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Model dimuat dari 07_output/model_transh.pt")
print(f"  dim={hp['dim']}  margin={hp['margin']}  C={hp['C']}")
print(f"  Best epoch: {checkpoint.get('best_epoch', '?')}")
print(f"  Best valid MRR: {checkpoint.get('best_valid_mrr', '?'):.4f}")

In [ ]:
# ── Load embeddings sebagai numpy (untuk operasi batch cepat) ──
ent_emb  = np.load(MODEL_DIR / "entity_embeddings.npy")
rel_emb  = np.load(MODEL_DIR / "relation_embeddings.npy")
norm_emb = np.load(MODEL_DIR / "hyperplane_normals.npy")

print(f"entity_embeddings : {ent_emb.shape}")
print(f"relation_embeddings: {rel_emb.shape}")
print(f"hyperplane_normals : {norm_emb.shape}")

# Normalisasi w_r (pastikan unit vector)
norm_emb = norm_emb / (np.linalg.norm(norm_emb, axis=-1, keepdims=True) + 1e-8)

---
## B. Load Data Film & Interaksi User

In [ ]:
# ── Metadata film ──
df_film = pd.read_csv(FILM_CSV, dtype=str)
film_info = {}  # show_id → {title, listed_in, ...}
for _, row in df_film.iterrows():
    sid = str(row["show_id"]).strip()
    film_info[sid] = {
        "title"     : str(row.get("title",     "")).strip(),
        "genre"     : str(row.get("listed_in", "")).strip(),
        "director"  : str(row.get("director",  "")).strip(),
        "rating"    : str(row.get("rating",    "")).strip(),
        "release_year": str(row.get("release_year", "")).strip(),
    }

print(f"Metadata film dimuat: {len(film_info):,} film")

In [ ]:
# ── Load split data ──
def load_tsv(filepath):
    triples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) == 3:
                triples.append(tuple(parts))
    return triples

train_triples = load_tsv(SPLIT_DIR / "train_triplets.tsv")
valid_triples = load_tsv(SPLIT_DIR / "valid_triplets.tsv")
test_triples  = load_tsv(SPLIT_DIR / "test_triplets.tsv")
all_triples   = train_triples + valid_triples + test_triples

# Kumpulkan film yang disukai tiap user (dari semua split)
user_liked_train = defaultdict(set)   # untuk filter film sudah ditonton
user_liked_test  = defaultdict(set)   # untuk evaluasi

for h, r, t in train_triples:
    if r == "liked" and h.startswith("U_"):
        user_liked_train[h].add(t)

for h, r, t in test_triples:
    if r == "liked" and h.startswith("U_"):
        user_liked_test[h].add(t)

# Semua show_id yang ada di KG (kandidat rekomendasi)
all_film_ids = sorted([
    e for e in entity2id.keys()
    if e.startswith("s") and e[1:].isdigit()
])
all_film_idx = np.array([entity2id[sid] for sid in all_film_ids])

# Semua user node
all_user_nodes = sorted([
    e for e in entity2id.keys()
    if e.startswith("U_")
])

print(f"Train triplets  : {len(train_triples):,}")
print(f"Test  triplets  : {len(test_triples):,}")
print(f"User dengan histori train: {len(user_liked_train):,}")
print(f"User dengan data test    : {len(user_liked_test):,}")
print(f"Kandidat film untuk rekomendasi: {len(all_film_ids):,}")

---
## C. Fungsi Rekomendasi

Menggunakan operasi numpy batch untuk efisiensi — menghitung skor untuk semua film sekaligus.

In [ ]:
LIKED_REL_ID = relation2id["liked"]

# Vektor relasi 'liked' yang sudah ditraining
d_liked = rel_emb[LIKED_REL_ID]       # d_r (translasi)
w_liked = norm_emb[LIKED_REL_ID]      # w_r (normal hiperplane, sudah dinormalisasi)

def project(e, w):
    """Proyeksikan embedding e ke hiperplane dengan normal w (numpy).
    e_⊥ = e − (e·w) w
    e bisa berupa (dim,) atau (N, dim).
    """
    if e.ndim == 1:
        return e - np.dot(e, w) * w
    else:
        dots = e @ w  # (N,)
        return e - np.outer(dots, w)

def l2_normalize(e):
    if e.ndim == 1:
        norm = np.linalg.norm(e)
        return e / (norm + 1e-8)
    else:
        norms = np.linalg.norm(e, axis=-1, keepdims=True)
        return e / (norms + 1e-8)


# Pre-compute proyeksi semua film ke hiperplane 'liked'
film_embs  = ent_emb[all_film_idx]                    # (N_film, dim)
film_perp  = l2_normalize(project(film_embs, w_liked)) # (N_film, dim)

print(f"Pre-compute proyeksi film: {film_perp.shape}")
print(f"Relasi 'liked' ID: {LIKED_REL_ID}")


def recommend_films(
    user_node: str,
    top_k: int = 10,
    exclude_seen: bool = True,
    seen_film_ids: set = None,
):
    """
    Rekomendasikan film untuk seorang user menggunakan TransH.

    Parameters
    ----------
    user_node    : string ID node user, misal 'U_42'
    top_k        : jumlah rekomendasi yang dikembalikan
    exclude_seen : jika True, filter film yang sudah ada di histori train user
    seen_film_ids: set show_id yang sudah ditonton (opsional, override default)

    Returns
    -------
    list of dict :
        [{'rank', 'show_id', 'score', 'title', 'genre', 'release_year'}, ...]
    """
    if user_node not in entity2id:
        return []

    # Embedding user
    user_emb  = ent_emb[entity2id[user_node]]           # (dim,)
    user_perp = l2_normalize(project(user_emb, w_liked)) # (dim,)

    # Score untuk semua film: ||user_⊥ + d_liked − film_⊥||²
    diff   = user_perp + d_liked - film_perp             # (N_film, dim)
    scores = np.linalg.norm(diff, axis=-1)               # (N_film,)

    # Filter film yang sudah ditonton
    if exclude_seen:
        seen = seen_film_ids if seen_film_ids is not None else user_liked_train.get(user_node, set())
        for i, sid in enumerate(all_film_ids):
            if sid in seen:
                scores[i] = 1e9  # exclude

    # Ambil top-K (skor terkecil = lebih relevan)
    top_idx = np.argsort(scores)[:top_k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        sid  = all_film_ids[idx]
        info = film_info.get(sid, {})
        results.append({
            "rank"        : rank,
            "show_id"     : sid,
            "score"       : float(scores[idx]),
            "title"       : info.get("title",        "—"),
            "genre"       : info.get("genre",        "—"),
            "release_year": info.get("release_year", "—"),
            "rating"      : info.get("rating",       "—"),
        })
    return results


print("Fungsi recommend_films() siap.")

### C1. Demo Rekomendasi — Contoh Beberapa User

In [ ]:
# Pilih user yang punya cukup data di test (untuk demo)
demo_users = sorted(
    user_liked_test.keys(),
    key=lambda u: len(user_liked_test[u]),
    reverse=True
)[:5]  # top-5 user paling aktif di test

TOP_K = 10

for user_node in demo_users:
    recs = recommend_films(user_node, top_k=TOP_K, exclude_seen=True)
    liked_in_test = user_liked_test[user_node]
    liked_in_train = user_liked_train.get(user_node, set())

    print(f"\n{'='*65}")
    print(f"  User: {user_node}")
    print(f"  Film disukai (train): {len(liked_in_train)}  |  Film di test: {len(liked_in_test)}")
    print(f"{'='*65}")
    print(f"  {'Rank':<5} {'show_id':<8} {'Score':>7}  {'Judul Film':<35} {'Hit?'}")
    print(f"  {'-'*70}")

    hit_count = 0
    for rec in recs:
        hit = "✓" if rec["show_id"] in liked_in_test else " "
        if rec["show_id"] in liked_in_test:
            hit_count += 1
        title_short = rec["title"][:33] + ".." if len(rec["title"]) > 35 else rec["title"]
        print(f"  {rec['rank']:<5} {rec['show_id']:<8} {rec['score']:>7.4f}  {title_short:<35} {hit}")

    precision = hit_count / TOP_K
    recall    = hit_count / max(len(liked_in_test), 1)
    print(f"  → Precision@{TOP_K}={precision:.3f}  Recall@{TOP_K}={recall:.3f}  "
          f"({hit_count}/{TOP_K} hit dari {len(liked_in_test)} film test)")

---
## D. Evaluasi Rekomendasi (User-Centric)

Metrik yang digunakan:
- **Precision@K**: proporsi rekomendasi yang benar di antara K teratas
- **Recall@K**: proporsi film test yang berhasil direkomendasikan
- **NDCG@K**: Normalized Discounted Cumulative Gain (memberi bobot lebih ke ranking tinggi)
- **Hit Rate@K**: apakah ada *setidaknya satu* film test di Top-K?

In [ ]:
def ndcg_at_k(recommended_ids, relevant_ids, k):
    """Hitung NDCG@k.
    recommended_ids: list show_id yang direkomendasikan (urutan skor)
    relevant_ids   : set show_id yang relevan (ground truth di test)
    """
    dcg = 0.0
    for i, sid in enumerate(recommended_ids[:k], start=1):
        if sid in relevant_ids:
            dcg += 1.0 / math.log2(i + 1)

    # Ideal DCG: semua item relevan ada di posisi 1, 2, 3, ...
    ideal_n = min(len(relevant_ids), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_n + 1))

    return dcg / idcg if idcg > 0 else 0.0


def evaluate_recommendations(user_nodes, k_list=(5, 10, 20), exclude_seen=True):
    """
    Evaluasi sistem rekomendasi untuk semua user yang punya ground truth di test.

    Returns
    -------
    df_per_user : DataFrame metrik per user
    summary     : dict metrik rata-rata
    """
    rows = []
    top_k_max = max(k_list)

    for user_node in user_nodes:
        relevant = user_liked_test.get(user_node, set())
        if not relevant:
            continue  # skip user tanpa ground truth di test

        recs = recommend_films(user_node, top_k=top_k_max, exclude_seen=exclude_seen)
        rec_ids = [r["show_id"] for r in recs]

        row = {"user": user_node, "n_test": len(relevant), "n_train": len(user_liked_train.get(user_node, set()))}
        for k in k_list:
            rec_k    = rec_ids[:k]
            hits_k   = sum(1 for sid in rec_k if sid in relevant)
            row[f"Precision@{k}"] = hits_k / k
            row[f"Recall@{k}"]    = hits_k / len(relevant)
            row[f"NDCG@{k}"]      = ndcg_at_k(rec_k, relevant, k)
            row[f"HitRate@{k}"]   = 1.0 if any(sid in relevant for sid in rec_k) else 0.0
        rows.append(row)

    df = pd.DataFrame(rows).set_index("user")

    # Ringkasan rata-rata
    metric_cols = [c for c in df.columns if any(m in c for m in ["Precision", "Recall", "NDCG", "HitRate"])]
    summary = df[metric_cols].mean().to_dict()

    return df, summary


print("Fungsi evaluate_recommendations() siap.")

In [ ]:
# Jalankan evaluasi untuk semua user yang punya data di test
K_LIST = (5, 10, 20)
eval_users = list(user_liked_test.keys())

print(f"Mengevaluasi {len(eval_users)} user (exclude_seen=True) ...")
df_eval, summary = evaluate_recommendations(eval_users, k_list=K_LIST, exclude_seen=True)

print(f"\nSelesai. {len(df_eval)} user dievaluasi.")
print("\n" + "─" * 50)
print("  Rata-rata Metrik Evaluasi Rekomendasi")
print("─" * 50)
for k in K_LIST:
    print(f"  K = {k}:")
    print(f"    Precision@{k}  : {summary.get(f'Precision@{k}', 0):.4f}")
    print(f"    Recall@{k}     : {summary.get(f'Recall@{k}', 0):.4f}")
    print(f"    NDCG@{k}       : {summary.get(f'NDCG@{k}', 0):.4f}")
    print(f"    HitRate@{k}    : {summary.get(f'HitRate@{k}', 0):.4f}")
    print()

display(df_eval.head(10))

---
## E. Simpan Output

In [ ]:
# Simpan evaluasi per user
df_eval_reset = df_eval.reset_index()
df_eval_reset.to_csv(OUT_DIR / "evaluasi_rekomendasi.csv", index=False, encoding="utf-8")
print(f"Evaluasi per user disimpan: evaluasi_rekomendasi.csv ({len(df_eval)} baris)")

In [ ]:
# Buat tabel rekomendasi untuk SEMUA user (Top-10)
TOP_K_SAVE = 10
all_recs = []

for user_node in all_user_nodes:
    recs = recommend_films(user_node, top_k=TOP_K_SAVE, exclude_seen=True)
    for rec in recs:
        all_recs.append({
            "user_node"   : user_node,
            "user_id"     : user_node.replace("U_", ""),
            "rank"        : rec["rank"],
            "show_id"     : rec["show_id"],
            "score"       : round(rec["score"], 6),
            "title"       : rec["title"],
            "genre"       : rec["genre"],
            "release_year": rec["release_year"],
            "rating"      : rec["rating"],
        })

df_recs = pd.DataFrame(all_recs)
df_recs.to_csv(OUT_DIR / "rekomendasi_per_user.csv", index=False, encoding="utf-8")

print(f"Rekomendasi disimpan: rekomendasi_per_user.csv")
print(f"  {len(all_user_nodes)} user × Top-{TOP_K_SAVE} = {len(df_recs):,} baris")
display(df_recs.head(15))

In [ ]:
# Simpan ringkasan evaluasi
summary_path = OUT_DIR / "evaluasi_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Evaluasi Sistem Rekomendasi TransH\n")
    f.write("=" * 50 + "\n")
    f.write(f"Jumlah user dievaluasi : {len(df_eval)}\n")
    f.write(f"Metrik (exclude_seen=True):\n\n")
    for k in K_LIST:
        f.write(f"  K = {k}:\n")
        f.write(f"    Precision@{k}  = {summary.get(f'Precision@{k}', 0):.4f}\n")
        f.write(f"    Recall@{k}     = {summary.get(f'Recall@{k}', 0):.4f}\n")
        f.write(f"    NDCG@{k}       = {summary.get(f'NDCG@{k}', 0):.4f}\n")
        f.write(f"    HitRate@{k}    = {summary.get(f'HitRate@{k}', 0):.4f}\n")
        f.write("\n")
print(f"Ringkasan disimpan: {summary_path}")

---
## F. Visualisasi Hasil Rekomendasi

In [ ]:
# Distribusi metrik evaluasi (boxplot)
metric_cols_k10 = [f"Precision@10", f"Recall@10", f"NDCG@10", f"HitRate@10"]
available_cols  = [c for c in metric_cols_k10 if c in df_eval.columns]

fig, axes = plt.subplots(1, len(available_cols), figsize=(16, 5))
fig.suptitle("Distribusi Metrik Evaluasi Rekomendasi (@K=10) per User",
             fontsize=13, fontweight="bold")

colors = ["#3B82F6", "#10B981", "#8B5CF6", "#F59E0B"]
for ax, col, color in zip(axes, available_cols, colors):
    data = df_eval[col].dropna()
    ax.boxplot(data, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.5),
               medianprops=dict(color="black", linewidth=2))
    ax.set_title(col, fontsize=11)
    ax.set_ylabel("Nilai")
    ax.set_xticks([])
    mean_val = data.mean()
    ax.axhline(mean_val, color="red", linestyle="--", linewidth=1.2, label=f"Mean={mean_val:.4f}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "viz_evaluasi_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: viz_evaluasi_boxplot.png")

In [ ]:
# Bar chart metrik rata-rata per K
metrics = ["Precision", "Recall", "NDCG", "HitRate"]
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))
fig.suptitle("Rata-rata Metrik Evaluasi Rekomendasi per K",
             fontsize=13, fontweight="bold")

bar_colors = ["#3B82F6", "#10B981", "#8B5CF6"]
for ax, metric in zip(axes, metrics):
    vals  = [summary.get(f"{metric}@{k}", 0) for k in K_LIST]
    bars  = ax.bar([str(k) for k in K_LIST], vals, color=bar_colors, edgecolor="white", width=0.5)
    ax.set_title(metric, fontsize=12)
    ax.set_xlabel("K")
    ax.set_ylim(0, max(vals) * 1.25 if vals else 1)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "viz_evaluasi_per_k.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: viz_evaluasi_per_k.png")

In [ ]:
# Top-10 film paling sering direkomendasikan
top_films_rec = (
    df_recs[df_recs["rank"] <= 10]
    .groupby(["show_id", "title"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .reset_index(name="frekuensi_rekomendasi")
)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(
    top_films_rec["title"].str[:45],
    top_films_rec["frekuensi_rekomendasi"],
    color="#3B82F6"
)
ax.set_title("Top-15 Film Paling Sering Direkomendasikan (di Top-10)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Frekuensi Muncul di Rekomendasi")
for bar, val in zip(bars, top_films_rec["frekuensi_rekomendasi"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=9)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "viz_top_films_recommended.png", dpi=150, bbox_inches="tight")
plt.show()
display(top_films_rec)
print("Tersimpan: viz_top_films_recommended.png")

In [ ]:
# Distribusi genre di rekomendasi vs film yang disukai di train
def extract_genres(df_col):
    """Ekspansi kolom genre (multi-nilai dipisah koma) menjadi list genre."""
    genres = []
    for val in df_col.dropna():
        for g in str(val).split(","):
            g = g.strip()
            if g and g != "nan":
                genres.append(g)
    return pd.Series(genres).value_counts()

# Genre di rekomendasi (top-10)
genre_rec  = extract_genres(df_recs[df_recs["rank"] <= 10]["genre"]).head(10)

# Genre di film yang disukai user (dari train)
train_liked_ids = set()
for _, liked_set in user_liked_train.items():
    train_liked_ids.update(liked_set)
genre_train = extract_genres(
    pd.Series([film_info.get(sid, {}).get("genre", "") for sid in train_liked_ids])
).head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Distribusi Genre: Film Disukai (Train) vs Film Direkomendasikan",
             fontsize=13, fontweight="bold")

axes[0].barh(genre_train.index[::-1], genre_train.values[::-1], color="#10B981")
axes[0].set_title("Film Disukai di Train", fontsize=12)
axes[0].set_xlabel("Frekuensi")
axes[0].grid(axis="x", alpha=0.3)

axes[1].barh(genre_rec.index[::-1], genre_rec.values[::-1], color="#3B82F6")
axes[1].set_title("Film Direkomendasikan (Top-10)", fontsize=12)
axes[1].set_xlabel("Frekuensi")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "viz_genre_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: viz_genre_distribution.png")

---
## G. Analisis Film Serupa (Film Similarity)

Berdasarkan cosine similarity di embedding space — film yang berdekatan di ruang embedding memiliki karakteristik serupa menurut model TransH.

In [ ]:
def find_similar_films(show_id: str, top_k: int = 10):
    """
    Cari film paling mirip berdasarkan cosine similarity di embedding space.
    """
    if show_id not in entity2id:
        print(f"show_id '{show_id}' tidak ditemukan di vocab.")
        return []

    idx    = entity2id[show_id]
    target = ent_emb[idx]                          # (dim,)
    
    # Cosine similarity dengan semua film
    norms  = np.linalg.norm(film_embs, axis=-1)    # (N_film,)
    sims   = (film_embs @ target) / (norms * np.linalg.norm(target) + 1e-8)

    # Exclude film itu sendiri
    self_pos = np.where(np.array(all_film_ids) == show_id)[0]
    if len(self_pos) > 0:
        sims[self_pos[0]] = -1.0

    top_idx = np.argsort(-sims)[:top_k]
    results = []
    for rank, i in enumerate(top_idx, start=1):
        sid  = all_film_ids[i]
        info = film_info.get(sid, {})
        results.append({
            "rank"    : rank,
            "show_id" : sid,
            "sim"     : float(sims[i]),
            "title"   : info.get("title", "—"),
            "genre"   : info.get("genre", "—"),
        })
    return results


# Demo: cari film mirip untuk beberapa film populer
demo_films = ["s341", "s574", "s602"]  # Inception, Kung Fu Panda, The Karate Kid

for sid in demo_films:
    info = film_info.get(sid, {})
    print(f"\n{'='*60}")
    print(f"Film query: {sid} — {info.get('title', '?')}")
    print(f"Genre     : {info.get('genre', '?')}")
    print(f"{'='*60}")
    similars = find_similar_films(sid, top_k=8)
    for s in similars:
        print(f"  [{s['rank']}] sim={s['sim']:.4f}  {s['show_id']:<8}  {s['title'][:40]:<40}  {s['genre'][:35]}")

---
## H. Ringkasan Akhir

In [ ]:
print("\n" + "=" * 60)
print("  RINGKASAN SISTEM REKOMENDASI TRANSH")
print("=" * 60)
print(f"  Model         : TransH (dim={hp['dim']}, margin={hp['margin']})")
print(f"  Entitas       : {n_entities:,}")
print(f"  Film tersedia : {len(all_film_ids):,}")
print(f"  User          : {len(all_user_nodes):,}")
print()
print("  Metrik Evaluasi (rata-rata, exclude_seen=True):")
for k in K_LIST:
    p  = summary.get(f'Precision@{k}', 0)
    r  = summary.get(f'Recall@{k}', 0)
    n  = summary.get(f'NDCG@{k}', 0)
    hr = summary.get(f'HitRate@{k}', 0)
    print(f"    @{k:<2}  P={p:.4f}  R={r:.4f}  NDCG={n:.4f}  HR={hr:.4f}")

print()
print("  Output files di 08_output/:")
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"    {f.name:<40} {size_kb:>8.1f} KB")
print("=" * 60)